# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR² Mass Spectrometry](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://mlcroissant.readthedocs.io/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL, and records, fields, and columns are always referenced by their `@id` as per the Croissant standard.

Dataset DOI: [10.71728/senscience.vq4a-28xa](https://sen.science/doi/10.71728/senscience.vq4a-28xa)


In [ ]:
# Ensure `mlcroissant` and plotting libraries are installed
!pip install mlcroissant matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata
md = dataset.metadata
print(f"Dataset: {md.name}\n\nDescription: {md.description}\n\nVersion: {md.version}\nLicense: {md.license}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

Entities are always referenced by their Croissant `@id`.


In [ ]:
import pprint

# List record sets available
record_sets = list(dataset.record_sets.keys())
print('Record Sets and their `@id`s:')
for rs_id in record_sets:
    print(f" - {rs_id}")

# For each record set, list fields and their `@id`s
for rs_id in record_sets:
    rs = dataset.record_sets[rs_id]
    print(f"\nRecord Set '{rs_id}' fields (by @id):")
    for field in rs.fields.values():
        print(f"   - {field.id}: {field.name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All uses of record set or field must reference their `@id`.


In [ ]:
# Extract data from each record set using @id
dataframes = {}

for rs_id in record_sets:
    print(f"Loading records from record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f" --> Records shape: {df.shape}")
        print(f"    - Columns (@id): {df.columns.tolist()}")
    else:
        print(" --> No data in this record set.")

# For demonstration, we'll select the first available record set with data
if dataframes:
    main_rs_id = list(dataframes.keys())[0]
    print(f'\nPrimary record set to work with: {main_rs_id}')
    display(dataframes[main_rs_id].head())
else:
    print('No record sets with data were found.')

## 4. Exploratory Data Analysis (EDA)
Apply exploratory data processing, referencing field names only by their `@id`.

We will choose a numeric field and a group/categorical field—update the following IDs as per the overview above. If you're using this notebook for another dataset, adapt the IDs accordingly.

In [ ]:
# ---- Specify field IDs for your EDA ----
rs_id = main_rs_id  # Use the record set chosen above
df = dataframes[rs_id]

# Preview available fields (by @id)
print('Available fields (@id):', df.columns.tolist())

# Choose a field @id for numeric analysis (e.g., 'coverage_percent', 'mw', 'normalized_abundance_control')
numeric_field_id = None
for col in df.columns:
    # Attempt to auto-select a field likely to be numeric, fallback to user input.
    if any(s in col.lower() for s in ['coverage', 'abundance', 'count', 'mw', 'quantity', 'percent']):
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
if not numeric_field_id:
    # If none found, default to first column that is numeric
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
if numeric_field_id is None:
    print("No numeric fields found!")

# Choose a group field (e.g., 'sample', 'protein_type', or other categorical field)
group_field_id = None
for col in df.columns:
    if any(s in col.lower() for s in ['sample', 'group', 'label', 'class', 'type', 'species', 'treatment']):
        if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col]):
            group_field_id = col
            break

print(f"Selected numeric field (@id): {numeric_field_id}")
print(f"Selected group field (@id): {group_field_id}")

# Filter numeric (e.g., abundance > threshold)
if numeric_field_id and numeric_field_id in df.columns:
    # Remove rows with NaN in numeric field
    df_num = df[~pd.isna(df[numeric_field_id])]
    threshold = np.percentile(df_num[numeric_field_id], 90)  # Top 10% as example
    filtered_df = df_num[df_num[numeric_field_id] > threshold]
    print(f"Filtered {len(filtered_df)} records with {numeric_field_id} > {threshold:.4g}:")
    display(filtered_df.head())

    # Normalize the chosen numeric column
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - df_num[numeric_field_id].mean()) / df_num[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by another field if available
    if group_field_id and group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        display(grouped.head())

## 5. Visualization
Visualize the distribution of the selected numeric field, segmented by the group field if available.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id and group_field_id in df.columns:
        # Limiting number of unique categories for plot clarity
        gcounts = df[group_field_id].value_counts()
        top_cats = gcounts.head(10).index
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=df[df[group_field_id].isin(top_cats)], x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id} (top 10 categories)")
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load and explore the FAIR² Mass Spectrometry dataset for Mast Cell Extracellular Vesicles using the Croissant standard and the `mlcroissant` library. 
- We listed available record sets and fields by their Croissant `@id`s.
- Loaded records into DataFrames for analysis.
- Filtered and normalized a protein property field, and visualized its distribution.

This workflow can be adapted to any Croissant-structured dataset by referencing entities using their `@id` fields. Further analysis can be performed according to your scientific questions!